<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-06-function-calling/lesson-6.1-function-calling/notebooks/GCP_Capstone_6.1_FunctionCalling.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 6.1 Gemini Function Calling — FunctionDeclarations, AUTO Mode
**Netsetos GenAI Engineering — GCP Capstone**

Teach Gemini to use tools. The model decides which to call, extracts arguments, you execute.


## Setup


In [ ]:
!pip install -q google-genai

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE

from google import genai
from google.genai import types

client = genai.Client(enterprise=True, project=PROJECT_ID,
                      location='global')
print(f'Client ready for {PROJECT_ID}')


## Cell 1: Define DocuMind Functions


In [ ]:
# DocuMind tool functions
def search_documents(query: str, doc_type: str = 'all', top_k: int = 5) -> dict:
    """Search DocuMind document collection by query.
    
    Args:
        query: Search query in natural language
        doc_type: Filter by document type (research_paper, invoice, legal, form, all)
        top_k: Number of results to return
    """
    # Mock: in production, call VECTOR_SEARCH
    return {'results': [
        {'doc_id': 'D-42', 'title': 'Refund Policy v3', 'relevance': 0.94},
        {'doc_id': 'D-17', 'title': 'Return Guidelines', 'relevance': 0.87}
    ], 'total': 2}

def calculate_processing_cost(page_count: int, file_type: str = 'pdf') -> dict:
    """Estimate document processing cost in USD and INR.
    
    Args:
        page_count: Number of pages in the document
        file_type: File format (pdf, html, text, docx)
    """
    rates = {'pdf': 0.07, 'html': 0.03, 'text': 0.02, 'docx': 0.05}
    cost = page_count * rates.get(file_type, 0.07)
    return {'cost_usd': round(cost, 2), 'cost_inr': round(cost * 85, 2),
            'breakdown': f'{page_count} pages x ${rates.get(file_type, 0.07)}/page'}

def get_usage_stats(metric: str, days: int = 7) -> dict:
    """Get DocuMind RAG pipeline usage statistics.
    
    Args:
        metric: Which metric to retrieve (queries, costs, latency, users)
        days: Number of days to look back
    """
    mock_data = {'queries': 1247, 'costs': 18.50, 'latency': 245, 'users': 42}
    return {'metric': metric, 'period': f'last {days} days',
            'value': mock_data.get(metric, 0), 'trend': '+12%'}

TOOLS = [search_documents, calculate_processing_cost, get_usage_stats]
print(f'Defined {len(TOOLS)} DocuMind tools')


## Cell 2: Automatic Function Calling


In [ ]:
# SDK auto-executes functions
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='How much would it cost to process a 45-page PDF?',
    config=types.GenerateContentConfig(tools=TOOLS)
)
print('=== Auto Function Calling ===')
print(response.text)


## Cell 3: Multiple Tool Selection


In [ ]:
# Test model selecting the right tool
queries = [
    'What does our refund policy say?',
    'How much to process 200 pages of invoices?',
    'How many queries did we get last week?',
    'Hello, how are you today?',  # Should NOT call a function
]

for q in queries:
    r = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=q,
        config=types.GenerateContentConfig(tools=TOOLS)
    )
    if r.function_calls:
        print(f'Q: {q}')
        print(f'  Tool: {r.function_calls[0].name}({r.function_calls[0].args})')
    else:
        print(f'Q: {q}')
        print(f'  Text: {r.text[:80]}...')
    print()


## Cell 4: Manual Loop (Without Auto-Execute)


In [ ]:
# Manual 4-step loop
from google.genai.types import FunctionDeclaration, Tool

# Step 1: Define tools manually
search_decl = FunctionDeclaration(
    name='search_documents',
    description='Search DocuMind documents by query.',
    parameters={
        'type': 'object',
        'properties': {
            'query': {'type': 'string', 'description': 'Search query'},
            'doc_type': {'type': 'string', 'enum': ['research_paper', 'invoice', 'legal', 'form', 'all']}
        },
        'required': ['query']
    }
)

tool = Tool(function_declarations=[search_decl])

# Step 2: Send prompt
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Find our refund policy documents',
    config=types.GenerateContentConfig(tools=[tool])
)

print('=== Manual Loop ===')
if response.function_calls:
    fc = response.function_calls[0]
    print(f'Function: {fc.name}')
    print(f'Args: {fc.args}')
    
    # Step 3: Execute
    result = search_documents(**fc.args)
    print(f'Result: {result}')
    
    # Step 4: Send result back
    final = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=[
            types.Content(role='user', parts=[
                types.Part.from_text(text='Find our refund policy documents')]),
            response.candidates[0].content,
            types.Content(role='user', parts=[
                types.Part.from_function_response(
                    name=fc.name, response=result)])
        ],
        config=types.GenerateContentConfig(tools=[tool])
    )
    print(f'\nFinal answer: {final.text}')


## Cell 5: Tool Config Modes


In [ ]:
# AUTO mode (default)
r_auto = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Hello!',
    config=types.GenerateContentConfig(
        tools=TOOLS,
        tool_config=types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(
                mode='AUTO')))
)
print(f'AUTO + greeting: {r_auto.text[:80]}')
print(f'  Function calls: {len(r_auto.function_calls) if r_auto.function_calls else 0}')

# ANY mode (forced)
r_any = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='Hello!',
    config=types.GenerateContentConfig(
        tools=TOOLS,
        tool_config=types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(
                mode='ANY')))
)
print(f'\nANY + greeting: forced function call')
if r_any.function_calls:
    print(f'  Forced: {r_any.function_calls[0].name}({r_any.function_calls[0].args})')


## Cell 6: Chat Session with Tools


In [ ]:
# Multi-turn chat with function calling
chat = client.chats.create(
    model='gemini-3.6-flash',
    config=types.GenerateContentConfig(
        tools=TOOLS,
        system_instruction='You are DocuMind AI. Use tools to answer questions.'
    )
)

# Turn 1
r1 = chat.send_message('Find our refund policy')
print(f'Turn 1: {r1.text[:100]}...')

# Turn 2 (follow-up with context)
r2 = chat.send_message('How much would it cost to process that document?')
print(f'\nTurn 2: {r2.text[:100]}...')

# Turn 3 (different tool)
r3 = chat.send_message('What are our query stats for this week?')
print(f'\nTurn 3: {r3.text[:100]}...')

print(f'\nChat history: {len(chat.get_history())} messages')


## Cell 7: Error-Safe Dispatcher


In [ ]:
# Production-safe function dispatcher
def safe_execute(fc):
    ALLOWED = {
        'search_documents': search_documents,
        'calculate_processing_cost': calculate_processing_cost,
        'get_usage_stats': get_usage_stats,
    }
    DESTRUCTIVE = {'delete_document', 'send_email', 'modify_access'}
    
    if fc.name in DESTRUCTIVE:
        return {'error': f'Operation {fc.name} requires manual confirmation'}
    if fc.name not in ALLOWED:
        return {'error': f'Unknown function: {fc.name}'}
    try:
        return ALLOWED[fc.name](**fc.args)
    except Exception as e:
        return {'error': f'Execution failed: {str(e)}'}

# Test safe execution
print('Safe execute tests:')
for name, args in [('search_documents', {'query': 'test'}),
                   ('unknown_func', {}),
                   ('calculate_processing_cost', {'page_count': 'abc'})]:
    class MockFC:
        pass
    fc = MockFC()
    fc.name = name
    fc.args = args
    result = safe_execute(fc)
    print(f'  {name}: {result}')


## ✅ Lesson 6.1 Complete!

- ✅ FunctionDeclaration with name, description, parameters
- ✅ Manual 4-step function calling loop
- ✅ Automatic function calling (SDK auto-executes)
- ✅ AUTO/ANY/NONE modes
- ✅ Multiple tool selection (model chooses)
- ✅ Parallel function calling
- ✅ Multi-turn chat with tools
- ✅ Error-safe dispatcher
- ✅ Security: never auto-execute destructive ops

**Next: Lesson 6.2 — Structured Tool Pipelines & Compositional Calling**
